# Dataset Sintético de Evasão Escolar

Este notebook gera um dataset sintético com base em variáveis relacionadas à evasão escolar no ensino público do Distrito Federal. As variáveis foram definidas previamente e incluem dados do estudante, família, comunidade, escola e professores.

# Definição de evasão escolar

- Precisamos de uma definição clara de evasão escolar. Exemplo, faltou mais de 30 dias seguidos?! 
- O ideal é seguir a definição do MEC, que considera evasão escolar a falta não justificada por mais de 15 dias consecutivos, conforme ChatGPT (temos que confirmar e pegar a fonte).
- Ou podemos considerar uma definição razoável usada em outros trabalhos acadêmicos.

# Propriedades de atributos

- Precisamos definir melhor as propriedades de cada atributo para facilitar a síntese de dados. Exemplo:
  - Sexo: variável categórica (masculino, feminino, "não informado" ?). Qual a proporção de cada categoria? Se não houver, podemos usar dados do IBGE.
  - Idade: variável numérica. Qual a média, mediana e desvio padrão? OU proporção entre esses, considerando uma variável categórica.

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random

# Configurações iniciais
n_alunos = 100000  # número de registros

In [22]:
sexos = ['Masculino', 'Feminino', 'Outro']
etapas = ['5º ano', '6º ano', '7º ano', '8º ano', '9º ano', '1ª série EM', '2ª série EM', '3ª série EM']
niveis_socioeconomicos = [f'N{i}' for i in range(1, 8)]
apoio_familiar_vals = ['Alto', 'Médio', 'Baixo']
comportamento_vals = ['Bom', 'Regular', 'Ruim']
aceitacao_vals = ['Alta', 'Média', 'Baixa']
expectativa_vals = ['Alta', 'Média', 'Baixa']
engajamento_vals = ['Alto', 'Moderado', 'Baixo']
infraestrutura_vals = ['Boa', 'Regular', 'Ruim']
material_vals = ['Completa', 'Parcial', 'Insuficiente']
flex_pedagogica_vals = ['Alta', 'Média', 'Baixa']
qualidade_pedagogica_vals = ['Boa', 'Regular', 'Ruim']

idades=[10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]

In [23]:
# Variáveis
proporcao_evasao = 0.1 # Supondo que 10% dos alunos evadem

pesos_idades=[1/13, 1/13, 1/13, 1/13, 1/13, 1/13, 1/13, 1/13, 1/13, 1/13, 1/13, 1/13, 1/13]
pesos_idades_evadiu=[0.5/13, 1.5/13, 1/13, 1/13, 1/13, 1/13, 1/13, 1/13, 1/13, 1/13, 1/13, 1/13, 1/13]

In [24]:
def gerar_idade(evadiu:bool):
    if evadiu:
        return random.choices(idades, weights=pesos_idades_evadiu)[0]
    else:
        return random.choices(idades, weights=pesos_idades)[0]

def gerar_renda():
    return round(np.random.lognormal(mean=6.5, sigma=0.5), 2)

def gerar_frequencia():
    return round(np.random.uniform(50, 100), 1)

def gerar_nota():
    return round(np.random.normal(6.5, 1.5), 1)

def gerar_distancia():
    return round(abs(np.random.normal(2, 1)), 1)

def gerar_sexo(evadiu:bool):
    if evadiu:
        return random.choices(sexos, weights=[0.48, 0.48, 0.04])[0]  # Evadidos têm menos diversidade de sexo
    else:
        return random.choices(sexos, weights=[0.48, 0.48, 0.04])[0]  # Proporção de não evadidos
    
def gerar_etapa_ensino(evadiu: bool):
    # todas as listas seguintes seguem essa ordem
    etapas = ["1° EF", "2° EF", "3° EF", "4° EF", "5° EF", "6° EF", \
              "7° EF", "8° EF", "9° EF", "1° EM", "2° EM", "3° EM"]

    # dados do censo escolar de 2023(ultimo ano disponivel c/ porcentagens de evasao)
    total_matriculas = 400686 - 48259       # total de matriculas - matriculas Edu. Infantil
    qtd_matriculas_etapas = [28087, 29163, 33253, 28054, 28682, 29944, \
                    30364, 28886, 30368, 31195, 29372, 25056]
    
    # calcula a distribuicao da proporcao de alunos por etapa
    distr_etapas = [i/total_matriculas for i in qtd_matriculas_etapas]

    percentual_evasao_etapa = [0.002, 0.001, 0.002, 0.002, 0.002, 0.01, \
                   0.014, 0.012, 0.018, 0.04, 0.046, 0.03]      # dados do censo escolar de 2023
    
    # calcula a distribuicao de abandonos
    evadidos_por_etapa = [qtd_matriculas_etapas[i] * percentual_evasao_etapa[i] for i in range(len(etapas))]
    total_evadidos = sum(evadidos_por_etapa)
    distr_etapas_evasao = [i/total_evadidos for i in evadidos_por_etapa]

    if evadiu:
        return random.choices(etapas, weights=distr_etapas)[0]
    else:
        return random.choices(etapas, weights=distr_etapas_evasao)[0]    

In [25]:

serie_evasao:list[bool] = [random.choices([True, False], weights=[proporcao_evasao, 1 - proporcao_evasao])[0] for _ in range(n_alunos) ]

# Criação do DataFrame
df = pd.DataFrame({
    'id_aluno': np.arange(1, n_alunos + 1),
    'evasao': serie_evasao,  # atributo que eu tenho e quero estimar
    'sexo': [gerar_sexo(evadir) for evadir in serie_evasao],
    'idade': [gerar_idade(evadir) for evadir in serie_evasao],
    # Novas variaveis
    'estudante_nis': np.random.choice([True, False], n_alunos, p=[0.3, 0.7]),
    'informou_nome_mae': np.random.choice([True, False], n_alunos, p=[0.95, 0.05]),
    'informou_nome_pai': np.random.choice([True, False], n_alunos, p=[0.7, 0.3]),
    'possui_deficiencia': np.random.choice([True, False], n_alunos, p=[0.05, 0.95]),
    'raca_cor': np.random.choice(
        ['Branca', 'Preta', 'Parda', 'Amarela', 'Indígena', 'Não declarada'],
        n_alunos,
        p=[0.35, 0.1, 0.45, 0.05, 0.02, 0.03]
    ),
    'tipo_localizacao_endereco_estudante': np.random.choice(
        ['Urbana', 'Rural'],
        n_alunos,
        p=[0.8, 0.2]
    ),
    'situacao_consolidada_no_ano': np.random.choice(
        ['Aprovado', 'Reprovado', 'Evadido', 'Em andamento'],
        n_alunos,
        p=[0.7, 0.15, 0.1, 0.05]
    ),

    'etapa_ensino': [gerar_etapa_ensino(i) for i in serie_evasao],
    'repetente': np.random.choice([True, False], n_alunos, p=[0.047, 0.953]),

    'frequencia_escolar': [gerar_frequencia() for _ in range(n_alunos)],
    'notas_medias': [gerar_nota() for _ in range(n_alunos)],
    'nivel_socioeconomico': np.random.choice(niveis_socioeconomicos, n_alunos),
    'renda_familiar': [gerar_renda() for _ in range(n_alunos)],
    
    'trabalha': np.random.choice([True, False], n_alunos, p=[0.15, 0.85]),
    'trabalho_domestico_excessivo': np.random.choice([True, False], n_alunos, p=[0.1, 0.9]),
    'problemas_de_saude': np.random.choice([True, False], n_alunos, p=[0.1, 0.9]),
    'violencia_domestica': np.random.choice([True, False], n_alunos, p=[0.05, 0.95]),
    'gravidez_na_adolescencia': np.random.choice([True, False], n_alunos, p=[0.03, 0.97]),
    'apoio_familiar': np.random.choice(apoio_familiar_vals, n_alunos, p=[0.4, 0.4, 0.2]),
    'comportamento_em_sala': np.random.choice(comportamento_vals, n_alunos, p=[0.5, 0.3, 0.2]),
    'aceitacao_pelos_pares': np.random.choice(aceitacao_vals, n_alunos, p=[0.5, 0.3, 0.2]),
    'necessita_assistencia_social': np.random.choice([True, False], n_alunos, p=[0.3, 0.7]),
    'expectativa_retorno': np.random.choice(expectativa_vals, n_alunos, p=[0.5, 0.3, 0.2]),
    'fator_desencadeador_presente': np.random.choice([True, False], n_alunos, p=[0.1, 0.9]),
    'distancia_da_escola_km': [gerar_distancia() for _ in range(n_alunos)],
    'violencia_na_comunidade': np.random.choice([True, False], n_alunos, p=[0.2, 0.8]),
    'engajamento_com_a_escola': np.random.choice(engajamento_vals, n_alunos, p=[0.4, 0.4, 0.2]),
    'infraestrutura_escolar': np.random.choice(infraestrutura_vals, n_alunos, p=[0.5, 0.3, 0.2]),
    'disponibilidade_material_didatico': np.random.choice(material_vals, n_alunos, p=[0.6, 0.3, 0.1]),
    'flexibilidade_pedagogica': np.random.choice(flex_pedagogica_vals, n_alunos, p=[0.4, 0.4, 0.2]),
    'qualidade_pedagogica_percebida': np.random.choice(qualidade_pedagogica_vals, n_alunos, p=[0.5, 0.3, 0.2])
})

# # Pontuação de risco
df['pontuacao_risco'] = (
     (100 - df['frequencia_escolar']) / 100 * 0.3 +
     (10 - df['notas_medias']) / 10 * 0.3 +
     df['repetente'].astype(int) * 0.1 +
     df['trabalha'].astype(int) * 0.1 +
     df['violencia_domestica'].astype(int) * 0.2
 )

df['pontuacao_risco'] = df['pontuacao_risco'] / df['pontuacao_risco'].max()

# # Evasão confirmada
df['evasao_confirmada'] = [np.random.rand() < r for r in df['pontuacao_risco']]


In [26]:
df.head()

,id_aluno,evasao,sexo,idade,estudante_nis,informou_nome_mae,informou_nome_pai,possui_deficiencia,raca_cor,tipo_localizacao_endereco_estudante,...,fator_desencadeador_presente,distancia_da_escola_km,violencia_na_comunidade,engajamento_com_a_escola,infraestrutura_escolar,disponibilidade_material_didatico,flexibilidade_pedagogica,qualidade_pedagogica_percebida,pontuacao_risco,evasao_confirmada
0,1,False,Feminino,11,False,True,True,False,Branca,Urbana,...,False,0.5,False,Moderado,Boa,Completa,Alta,Boa,0.216064,False
1,2,False,Masculino,19,True,True,True,False,Parda,Urbana,...,False,0.7,False,Alto,Regular,Parcial,Alta,Boa,0.372110,True
2,3,False,Feminino,22,False,True,True,False,Amarela,Urbana,...,False,2.3,False,Moderado,Regular,Insuficiente,Baixa,Ruim,0.091583,False
3,4,False,Masculino,19,False,True,True,False,Parda,Urbana,...,False,4.0,False,Alto,Boa,Parcial,Alta,Ruim,0.328097,True
4,5,False,Masculino,16,False,True,True,False,Branca,Urbana,...,False,1.1,False,Moderado,Ruim,Completa,Média,Boa,0.170717,False


In [27]:
# Visualização da distribuição da pontuação de risco
# plt.hist(df['pontuacao_risco'], bins=20, edgecolor='black')
# plt.title('Distribuição da Pontuação de Risco')
# plt.xlabel('Pontuação de risco')
# plt.ylabel('Quantidade de alunos')
# plt.show()

In [28]:
# Salvar dataset em CSV
df.to_csv('dataset_evasao_sintetico.csv', index=False)
print('Dataset salvo como dataset_evasao_sintetico.csv')

Dataset salvo como dataset_evasao_sintetico.csv
